# Lecture 30: Practice Problems Master Solutions Notebook
## Supervised Machine Learning (Part 6) — Boosting & Advanced Ensembles

This notebook provides verified production solutions and unit tests for the **3 Core Case Studies** in Lecture 30.

### Case Study 1: NovaGen Research Labs — Clinical Patient Triage (Solutions)
Evaluating Logistic Regression, KNN, Random Forest, Gradient Boosting, and Soft Voting Classifier on NovaGen healthcare data.

In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import recall_score, accuracy_score

csv_path = r'c:\Users\Shivam Keshari\OneDrive\Desktop\AI-ML\lecture_30\novagen_dataset.csv'
df = pd.read_csv(csv_path)
X = df.drop('Target', axis=1)
y = df['Target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# 1. Logistic Regression
lr = LogisticRegression(penalty='l2', solver='liblinear', max_iter=1000)
lr.fit(X_train_s, y_train)
lr_rec = recall_score(y_test, lr.predict(X_test_s))

# 2. KNN
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_s, y_train)
knn_rec = recall_score(y_test, knn.predict(X_test_s))

# 3. Random Forest (on unscaled/scaled)
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
rf_rec = recall_score(y_test, rf.predict(X_test))

# 4. Gradient Boosting
gb = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
gb.fit(X_train, y_train)
gb_rec = recall_score(y_test, gb.predict(X_test))

# 5. Soft Voting Classifier
voting_clf = VotingClassifier(
    estimators=[
        ('lr', LogisticRegression(solver='liblinear', max_iter=1000)),
        ('knn', KNeighborsClassifier(n_neighbors=5)),
        ('dt', DecisionTreeClassifier(max_depth=5, random_state=42))
    ],
    voting='soft'
)
voting_clf.fit(X_train_s, y_train)
vote_rec = recall_score(y_test, voting_clf.predict(X_test_s))

print(f'Logistic Regression Recall : {lr_rec * 100:.2f}%')
print(f'KNN Recall                 : {knn_rec * 100:.2f}%')
print(f'Voting Classifier Recall   : {vote_rec * 100:.2f}%')
print(f'Gradient Boosting Recall   : {gb_rec * 100:.2f}%')
print(f'Random Forest Recall       : {rf_rec * 100:.2f}%')

assert rf_rec > 0.90, 'Random Forest must achieve clinical recall > 90%'
assert gb_rec > 0.90, 'Gradient Boosting must achieve clinical recall > 90%'
print('Case Study 1 Passed!')


Logistic Regression Recall : 83.76%
KNN Recall                 : 86.93%
Voting Classifier Recall   : 91.09%
Gradient Boosting Recall   : 92.48%
Random Forest Recall       : 93.96%
Case Study 1 Passed!


### Case Study 2: Adaptive Sample Reweighting in AdaBoost (Solutions)
Mathematical verification of estimator weight formula and training AdaBoostClassifier.

In [2]:
from sklearn.datasets import make_classification
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# 1. Mathematical calculation of alpha_m
def compute_alpha(eps):
    return 0.5 * np.log((1.0 - eps) / (eps + 1e-12))

alpha_10 = compute_alpha(0.10)
alpha_50 = compute_alpha(0.50)
print(f'Estimator weight for error eps=0.10: alpha = {alpha_10:.4f}')
print(f'Estimator weight for error eps=0.50: alpha = {alpha_50:.4f}')
assert round(alpha_50, 4) == 0.0, 'Random guess model must receive 0 voting power'
assert alpha_10 > 1.0, 'Low-error model must receive high voting power'

# 2. Train AdaBoost on synthetic data
X_syn, y_syn = make_classification(n_samples=500, n_features=20, n_informative=10, random_state=42)
X_tr_ad, X_te_ad, y_tr_ad, y_te_ad = train_test_split(X_syn, y_syn, test_size=0.25, random_state=42)

abc = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1),
    n_estimators=50,
    learning_rate=0.5,
    random_state=42
)
abc.fit(X_tr_ad, y_tr_ad)
ada_acc = accuracy_score(y_te_ad, abc.predict(X_te_ad))
print(f'AdaBoost (50 Stumps) Test Accuracy: {ada_acc * 100:.2f}%')
assert ada_acc > 0.75, 'AdaBoost must combine weak stumps into a strong learner'
print('Case Study 2 Passed!')


Estimator weight for error eps=0.10: alpha = 1.0986
Estimator weight for error eps=0.50: alpha = -0.0000
AdaBoost (50 Stumps) Test Accuracy: 79.20%
Case Study 2 Passed!


### Case Study 3: Meta-Learning Architectures — Stacking vs Blending (Solutions)
Implementing StackingClassifier and verifying that meta-learning improves predictive accuracy over individual base models.

In [3]:
from sklearn.ensemble import StackingClassifier

# Base models
rf_base = RandomForestClassifier(n_estimators=50, random_state=42)
gb_base = GradientBoostingClassifier(n_estimators=50, max_depth=3, random_state=42)

rf_base.fit(X_tr_ad, y_tr_ad)
gb_base.fit(X_tr_ad, y_tr_ad)

rf_acc = accuracy_score(y_te_ad, rf_base.predict(X_te_ad))
gb_acc = accuracy_score(y_te_ad, gb_base.predict(X_te_ad))

# Stacking Meta-Learner
stacking_clf = StackingClassifier(
    estimators=[
        ('rf', RandomForestClassifier(n_estimators=50, random_state=42)),
        ('gb', GradientBoostingClassifier(n_estimators=50, max_depth=3, random_state=42))
    ],
    final_estimator=LogisticRegression(),
    cv=5
)
stacking_clf.fit(X_tr_ad, y_tr_ad)
stack_acc = accuracy_score(y_te_ad, stacking_clf.predict(X_te_ad))

print(f'Base Model 1 (Random Forest)     : {rf_acc * 100:.2f}%')
print(f'Base Model 2 (Gradient Boosting) : {gb_acc * 100:.2f}%')
print(f'Stacking Meta-Learner Accuracy   : {stack_acc * 100:.2f}%')

assert stack_acc >= min(rf_acc, gb_acc), 'Stacking must perform at least as well as individual models'
print('Case Study 3 Passed!')
print('All 3 Lecture 30 Case Studies Verified Successfully!')


Base Model 1 (Random Forest)     : 87.20%
Base Model 2 (Gradient Boosting) : 87.20%
Stacking Meta-Learner Accuracy   : 88.00%
Case Study 3 Passed!
All 3 Lecture 30 Case Studies Verified Successfully!
